## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

[Langchain Builtin Middlewares](https://docs.langchain.com/oss/python/langchain/middleware/built-in)


In [3]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

### Message Based summarization
agent = create_agent(
    model=("google_genai:gemini-3-flash-preview"),
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger= ("messages",10),
            keep=("messages", 4)
            )
    ]
)

In [5]:
### Run with a thread id
config = {
    "configurable": {"thread_id": "test-1"}
}

In [6]:
questions = [
    "What is 3+3?",
    "What is 1*4?",
    "What is 5*6?",
    "What is 5/2?",
    "What is 9*9?",
    "What is 69-0?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(f"Messages : {response}")
    print(f"Length of Messages : {len(response["messages"])}")

Messages : {'messages': [HumanMessage(content='What is 3+3?', additional_kwargs={}, response_metadata={}, id='7548cda1-39ff-432d-a80f-790e56d1b5e3'), AIMessage(content=[{'type': 'text', 'text': '3 + 3 = 6', 'extras': {'signature': 'EoQBCoEBAb4+9vtnHP+3OtuoRikmI2gO8/VZfvPSoJB0iA4ogVdApwgYx+PMW6pBPcuwq9so/6DI/cr++zuuAGyKhdy3mRlD3AyJh4W/5ChImkEFQl+/k7dldywvf6McQk/o9y9pBRcn1jjxHjx+qp6JUnUpG/xw0cuRLdSYXEUFJmvRxAMj'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d41bf-7cbd-7fa2-a441-8ad1deb5c38f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 36, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 29}})]}
Length of Messages : 2
Messages : {'messages': [HumanMessage(content='What is 3+3?', additional_kwargs={}, response_metadata={}, id='7548cda1-39ff-432d-a80f-790e56d

In [10]:
### Summarization triggered with token size
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city, hardcoded to return a long response that uses more tokens"""
    return f""" Hotels in {city}:
    1. Grand Hyatt - 4 star, 300$/night, pool, gym, bar
    2. Hilton - 5 star, 390$/night, pool, gym, bar
    3. Sheraton - 5 star, 500$/night, pool, gym, bar
    4. Grand Hyatt - 4 star, 300$/night, pool, gym, bar
    5. Hilton - 5 star, 390$/night, pool, gym, bar
    6. Sheraton - 5 star, 500$/night, pool, gym, bar"""

agent_tokens = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",500),
            keep=("tokens",200),
        )
    ]
)

In [11]:
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

cities = ["Paris", "Tokyo", "Mumbai", "New York", "London"]

for city in cities:
    response = agent_tokens.invoke(
        {"messages": [
            HumanMessage(content=f"Find hotels for city : {city}")
        ]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} has ~ {tokens} tokens, {len(response["messages"])} messages")
    print(response['messages'])

Paris has ~ 159 tokens, 4 messages
[HumanMessage(content='Find hotels for city : Paris', additional_kwargs={}, response_metadata={}, id='c09889c1-84aa-45cb-b6b5-28c45a693e1e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'7abbdeba-c272-43e9-8779-d7b99327cb0d': 'EooDCocDAb4+9vsXaQOnUP359Iijl6zNHgAnTRp8F4CIayG16qcSs8rVEMgsWAADMMzc4c/RAJ5n3QgZ0ubfTk7XhCSu1sgSuhiILeJA2Mu1L4Lac0Qk2fCkXM7KREFIRGRegTw9X4IZCrU/vcJfITw5AxNoPE96t/Qk3a2a1BjG+agl8SMmiQoUsXdkVKNDXXsSZqN0LhFhihtkead/3+U4vbjsuQh3oBd/EMMQim9P3a17DIaOOwjTIs27dKX7AlDomfWEkPiROO5Lg2eC5+pBdSCfaAoxzZU8bVU6rv8S7ozTLl0XCRkFVR7H65EhAogBU4zJl7d6L4p+7wf0DtcNC/NCsEZYPjAaaCiR0jcbF2RmCYNKw7qBF0UnUldDxhWljg9Fhu/9ey3uIGrn3/jd8ftelgGWgyDa3fHy/HiwRZW5yKY6E7Xa9Nih6H4Py6lD4248ngs6kydc7+6zWtLZQDHLxkGanNww54lvj5a950VskGkybJTvB6StqqAEC1SLQHeGdQcjRCfI4g=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-prev

### Human In the Loop Middleware
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. 
Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.
